# 실습 2 목둘레로 허리둘레를 예측하는 선형 회귀 모델 구현하기 (교사용)

교과서 98~101쪽, 선택 활동 2(103쪽)

**문제 상황**: 목둘레로 내 허리 치수에 잘 맞는 청바지를 고를 수 있다고 한다. 신체 치수로 허리둘레를 예측하는 선형 회귀 모델을 만들어 보자.

| 단계 | 하는 일 |
|---|---|
| 단계 1 문제 정의하기 | 신체 치수를 알면 허리둘레를 예측할 수 있을까? |
| 단계 2 데이터 수집 및 전처리하기 | 데이터 불러오기, 독립 변수·종속 변수 정하기, 훈련·테스트 데이터 나누기 |
| 단계 3 모델 생성하기 | 선형 회귀 모델 학습, 회귀계수·절편·결정계수 구하기 |
| 단계 4 모델 평가하기 | 테스트 데이터로 결정계수와 오차 구하기, 예측해 보기 |

**데이터**: 교과서 99쪽 링크의 `jeans.csv`(공군 신체 측정 정보, 963명: neck, weight, heap, waist). 선생님이 나눠 준 `청소년_신체측정정보_16세_19세.csv`로도 할 수 있습니다.

**사용 방법**: 위에서부터 차례로 ▶ 버튼을 눌러 실행합니다. 오른쪽의 선택 상자와 슬라이더로 설정을 바꿔 다시 실행해 보세요.

### 교사용 메모
- **데이터**: 교과서 `jeans.csv`는 공군 신체 측정 정보(남, 여)에서 4개 열만 남기고 결측치·이상치를 지운 963행입니다. 엉덩이둘레의 열 이름 `heap`은 교과서 그대로입니다(영어로는 hip이 맞습니다).
- **선생님 자료**: `청소년_신체측정정보_16세_19세.csv`(253행)를 고르면 열 이름을 교과서와 같게 바꾸고, 999.9처럼 잘못 들어간 값을 IQR 기준으로 지운 뒤 같은 코드로 진행합니다. 이 파일은 저장 방식에 따라 UTF-8 또는 cp949이고, 엉덩이둘레 열 이름이 '엉덩이둘레_츠'로 깨져 있을 수 있어 두 경우를 모두 처리합니다.
- **결정계수 값**: 교과서 jeans.csv로 이 노트북을 실행하면 회귀계수 2.8324, 절편 −24.7212, 훈련 0.698, 테스트 0.713(단순), 다중 0.856이 나옵니다(교과서와 같음). 선생님 자료로는 단순 약 0.48, 다중 약 0.81입니다.
- **회귀계수 비교 주의**: 교과서 103쪽은 '목둘레 회귀계수가 가장 크므로 가장 큰 영향'이라고 설명하지만, 단위와 범위가 다른 변수의 계수를 그대로 비교하면 안 됩니다. 정규화한 뒤 비교하거나, 변수를 하나씩 빼 보며 결정계수가 얼마나 떨어지는지 보는 방법을 함께 안내해 주세요. 아래 칸이 그 비교입니다. jeans.csv로 표준화해 비교하면 엉덩이둘레(4.65) > 목둘레(2.54) > 몸무게(1.34) 순입니다.

In [ ]:
#@title 데이터 준비하기
how = "사이트에서 바로 받기" #@param ["사이트에서 바로 받기", "내 컴퓨터에서 올리기"]
#@markdown 디지털 교과서에 올려 둔 교과서 자료(jeans.csv)를 코랩으로 바로 가져옵니다. 직접 모은 파일을 쓰려면 '내 컴퓨터에서 올리기'를 고르세요.
if how.startswith('사이트'):
    import urllib.request
    for name in ['jeans.csv']:
        urllib.request.urlretrieve('https://aibook-dl.vercel.app/data/' + name, name)
        print('받았습니다:', name)
else:
    from google.colab import files
    uploaded = files.upload()


In [ ]:
#@title 단계 2 (1) 데이터 불러오기 (교과서 99쪽)
data_file = "jeans.csv (교과서)" #@param ["jeans.csv (교과서)", "청소년_신체측정정보_16세_19세.csv (선생님 자료)"]
import pandas as pd

if data_file.startswith('jeans'):
    df = pd.read_csv('jeans.csv')                       # 데이터 프레임(df) 생성하기
else:
    name = '청소년_신체측정정보_16세_19세.csv'
    try:
        raw = pd.read_csv(name, encoding='cp949')
    except UnicodeDecodeError:
        raw = pd.read_csv(name, encoding='utf-8-sig')
    raw.columns = [c.strip().replace('_츠', '_cm') for c in raw.columns]
    # 교과서와 같은 열 이름으로 바꾸기
    df = raw.rename(columns={'목둘레_cm': 'neck', '몸무게_kg': 'weight', '엉덩이둘레_cm': 'heap', '허리둘레_cm': 'waist'})
    df = df[['neck', 'weight', 'heap', 'waist']]
    # 이 자료에는 999.9처럼 잘못 입력된 값(이상치)이 있어 IQR 기준으로 지운다
    keep = pd.Series(True, index=df.index)
    for c in df.columns:
        q1, q3 = df[c].quantile(0.25), df[c].quantile(0.75)
        keep &= df[c].between(q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1))
    print(f'이상치가 있는 {(~keep).sum()}개 행을 지웠습니다.')
    df = df[keep]

df.info()
df.head()

In [ ]:
#@title (2) 독립 변수와 종속 변수 정하기, 데이터 나누기 (교과서 99~100쪽)
import numpy as np
X = df['neck'].values.reshape(-1, 1)        # 독립 변수(X): 목둘레 (2차원 배열로)
y = df['waist'].values                      # 종속 변수(y): 허리둘레

from sklearn.model_selection import train_test_split
test_size = 0.3 #@param {type:"slider", min:0.1, max:0.5, step:0.05}
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)
print('훈련 데이터:', X_train.shape, y_train.shape)
print('테스트 데이터:', X_test.shape, y_test.shape)

### 예측해 보기
목둘레만으로 허리둘레를 얼마나 잘 설명할 수 있을까요? 결정계수(R²)가 몇쯤 나올지 먼저 예측해 보세요.

In [ ]:
#@title 단계 3 모델 생성하기: 선형 회귀 모델 학습 (교과서 100쪽)
from sklearn.linear_model import LinearRegression
model = LinearRegression()                  # 모델 생성하기
model.fit(X_train, y_train)                 # 모델 학습하기

print('회귀계수:', model.coef_, ' 절편:', model.intercept_)
print(f'선형 회귀 식: 허리둘레 = {model.coef_[0]:.4f} × 목둘레 + ({model.intercept_:.4f})')
print(f'훈련 데이터 결정계수(R²): {model.score(X_train, y_train):.3f}')

In [ ]:
#@title 단계 4 모델 평가하기 (교과서 101쪽)
from sklearn.metrics import mean_absolute_error, mean_squared_error
predictions = model.predict(X_test)         # 테스트 데이터로 예측하기
print(f'테스트 데이터 결정계수(R²): {model.score(X_test, y_test):.3f}')
mse = mean_squared_error(y_test, predictions)
print(f'MAE(평균절대오차): {mean_absolute_error(y_test, predictions):.2f}')
print(f'MSE(평균제곱오차): {mse:.2f}   RMSE(평균제곱근오차): {np.sqrt(mse):.2f}')
print()
print('예측값       :', predictions[:5].round(1))
print('테스트 결괏값:', y_test[:5])

import matplotlib.pyplot as plt
plt.scatter(X_test, y_test, color='blue')   # 테스트 데이터 실젯값(산점도)
plt.plot(X_test, predictions, color='red')  # 예측 결과(선그래프)
plt.xlabel('neck'); plt.ylabel('waist')
plt.show()

In [ ]:
#@title 내 목둘레로 허리둘레 예측하기
my_neck = 36 #@param {type:"slider", min:28, max:48, step:0.5}
print(f'목둘레 {my_neck}cm → 예측 허리둘레 {model.predict([[my_neck]])[0]:.1f}cm')

In [ ]:
#@title 선택 활동 2 다중 선형 회귀로 허리둘레 예측하기 (교과서 103쪽)
features = ['neck', 'weight', 'heap']       # 독립 변수: 목둘레, 몸무게, 엉덩이둘레
X2 = df[features].values
y2 = df['waist'].values
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=test_size, random_state=42)
model2 = LinearRegression().fit(X2_train, y2_train)

print('회귀계수:', dict(zip(features, model2.coef_.round(4))), ' 절편:', round(model2.intercept_, 4))
terms = ' + '.join(f'{w:.3f}×{c}' for w, c in zip(model2.coef_, features))
print(f'선형 회귀 식: waist = {terms} + ({model2.intercept_:.3f})')
print(f'테스트 데이터 결정계수(R²): 단순 {model.score(X_test, y_test):.3f} → 다중 {model2.score(X2_test, y2_test):.3f}')

In [ ]:
#@title [교사용] 정규화한 뒤 회귀계수 비교하기
from sklearn.preprocessing import StandardScaler
sc = StandardScaler().fit(X2_train)
m_std = LinearRegression().fit(sc.transform(X2_train), y2_train)
print('표준화한 회귀계수(1 표준편차가 늘 때 허리둘레 변화, cm):')
for c, w in sorted(zip(features, m_std.coef_), key=lambda t: -abs(t[1])):
    print(f'  {c:7s} {w:6.2f}')

### 정리하기
1. 단순 선형 회귀와 다중 선형 회귀의 테스트 결정계수를 비교해 보세요. 어느 쪽이 더 잘 설명하나요?
2. 회귀계수의 크기만 보고 '가장 영향이 큰 변수'를 정해도 될까요? (힌트: 변수마다 단위와 범위가 다릅니다.)